# 2. Linear regression on house dataset
- Use this dataset on house price prediction in kaggle, 
- perform EDA on it and then predict the house prices using linear regression.
- You can drop the categorical features and the date columns. 
- Then try ElasticNetCV and RidgeCV on the same dataset. 
- Record the scores in a dataframe with the columns mae, mse, rmse so you can compare the models.

### EDA

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/data.csv")#) # make date column index and parse dates

df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df = df.drop(["date", "street", "city", "statezip", "country"], axis="columns")
df.head()

# inspecting values looking for 0 and extreme outliers

In [ ]:
df["price"].sort_values().min()

In [ ]:
df.shape

In [ ]:
# remove all 0 vals in price
df = df[df["price"] > 0]

In [ ]:
# checking low vals
df["price"].sort_values(ascending=True).head()

In [ ]:
# checking high
df["price"].sort_values(ascending=False).head()

### options for further testing with extreme outliers filtered

In [ ]:
# # calculating upper and lower limits by standard deviations and filtering out the outliers
# lower_limit = df["price"].mean() - 3 * df["price"].std()
# upper_limit = df["price"].mean() + 3 * df["price"].std()

# df = df[(df["price"] >= lower_limit) & (df["price"] <= upper_limit)]

# print(f"rows after filtering: {df.shape[0]} rows")

In [ ]:
# # calculating upper and lower limits by percentiles and filtering out the outliers
# lower_limit = df["price"].quantile(0.01)
# upper_limit = df["price"].quantile(0.99)

# df = df[(df["price"] >= lower_limit) & (df["price"] <= upper_limit)]

# print(f"rows after filtering: {df.shape[0]} rows")
# print(f"lower : {lower_limit:,.0f}  |  upper: {upper_limit:,.0f}")


In [ ]:
df.shape

In [ ]:
df.describe()

### divide X, y

In [ ]:
X, y = df.drop("price", axis = "columns"), df["price"]
X.head()

In [ ]:
y.head()

### Correlation heatmap for feat selection/ feat engineering
- checking correlations to determine what feats to eventually drop
- ex. drop those with high or low corr

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig,ax = plt.subplots(1, figsize=(16,8), dpi=150)

sns.heatmap(df.corr(), annot=True)

In [ ]:
# merge X and y temporary
df_with_price = X.copy()
df_with_price["price"] = y

# Calc correlations. sort by price
corr_price = df_with_price.corr(numeric_only=True)[["price"]].sort_values(
    by="price", ascending=False
)

# visualize
plt.figure(figsize=(6, 8))
sns.heatmap(corr_price, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("correlation with price", fontsize=16)
plt.show()

## Linear regression

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

X_train.shape, y_train.shape

### testing 2 types of scaling: MinMax and standard

In [ ]:
# from sklearn.preprocessing import MinMaxScaler

# scaler = MinMaxScaler()
# scaler.fit(X_train)
# scaled_X_train = scaler.transform(X_train) 
# scaled_X_test = scaler.transform(X_test) 

# scaled_X_train.min(), scaled_X_train.max(), scaled_X_test.min(), scaled_X_test.max()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
scaled_X_train = scaler.transform(X_train) 
scaled_X_test = scaler.transform(X_test) 

scaled_X_train.min(), scaled_X_train.max(), scaled_X_test.min(), scaled_X_test.max()

In [ ]:
from sklearn.linear_model import LinearRegression

model= LinearRegression()
model

In [ ]:
model.fit(scaled_X_train, y_train)
model.coef_

In [ ]:
model.intercept_

## Linear - prediction

In [ ]:
test_sample_features = scaled_X_test[0].reshape(1, -1)
test_sample_target = y_test.values[0]
test_sample_features, test_sample_target


In [ ]:
test_sample_features.shape

In [ ]:
test_sample_target

In [ ]:
model.predict(test_sample_features)

In [ ]:
test_sample_target

### prediction on test data

In [ ]:
y_pred = model.predict(scaled_X_test)
y_pred.shape

In [ ]:
y_test.shape

In [ ]:
y_pred[:5]

In [ ]:
y_test[:5].values

### evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def metrics(y_test, y_pred):
    MSE = mean_squared_error(y_test, y_pred)
    RMSE = np.sqrt(MSE)
    MAE = mean_absolute_error(y_test, y_pred)

    return {"MSE": MSE, "MAE": MAE, "RMSE": RMSE}

metrics(y_test, y_pred)

### ridge_regression

In [ ]:
from sklearn.linear_model import ElasticNetCV, RidgeCV

model_ridgeCV = RidgeCV(alphas=[.0001, .001, .01, .1, .5, 1, 5, 10], scoring="neg_mean_squared_error")
model_ridgeCV.fit(scaled_X_train, y_train)

model_ridgeCV.alpha_

In [ ]:
y_pred = model_ridgeCV.predict(scaled_X_test)
metrics(y_test, y_pred)

In [ ]:
model_ridgeCV.coef_

### lasso regression

In [ ]:
from sklearn.linear_model import LassoCV
alphas = np.logspace(-3, 3, 50)
model_lassoCV = LassoCV(alphas=alphas, cv=5, max_iter=10000)
model_lassoCV.fit(scaled_X_train, y_train)

model_lassoCV.alpha_

In [ ]:
y_pred = model_lassoCV.predict(scaled_X_test)

metrics(y_test, y_pred)

In [ ]:
model_lassoCV.coef_

### elastic NET regression

In [ ]:
model_elastic = ElasticNetCV(
    l1_ratio=[.1, .5, .7, .9, .95, .99, 1], alphas=alphas, max_iter=10000
)
model_elastic.fit(scaled_X_train, y_train)
model_elastic.l1_ratio_

In [ ]:
model_elastic.alpha_

In [ ]:
y_pred = model_elastic.predict(scaled_X_test)
metrics(y_test, y_pred)

In [ ]:
model_elastic.coef_

### Record scores in DataFrame

In [ ]:
# Calc errors for each model
mae_lin, mse_lin, rmse_lin = metrics(y_test, model.predict(scaled_X_test)).values()
mae_ridge, mse_ridge, rmse_ridge = metrics(y_test, model_ridgeCV.predict(scaled_X_test)).values()
mae_lasso, mse_lasso, rmse_lasso = metrics(y_test, model_lassoCV.predict(scaled_X_test)).values()
mae_elastic, mse_elastic, rmse_elastic = metrics(y_test, model_elastic.predict(scaled_X_test)).values()

# Create compare DataFrame
score_df = pd.DataFrame({
    "Model": ["Linear", "Ridge", "Lasso", "ElasticNet"],
    "MAE": [mae_lin, mae_ridge, mae_lasso, mae_elastic],
    "MSE": [mse_lin, mse_ridge, mse_lasso, mse_elastic],
    "RMSE": [rmse_lin, rmse_ridge, rmse_lasso, rmse_elastic]
})

print(score_df)

In [ ]:
# bar chart for scores
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MAE
axes[0].bar(score_df["Model"], score_df["MAE"], color="skyblue")
axes[0].set_title("MAE per modell")
axes[0].set_ylabel("MAE")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

# MSE
axes[1].bar(score_df["Model"], score_df["MSE"], color="lightgreen")
axes[1].set_title("MSE per modell")
axes[1].set_ylabel("MSE")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)

# RMSE
axes[2].bar(score_df["Model"], score_df["RMSE"], color="salmon")
axes[2].set_title("RMSE per modell")
axes[2].set_ylabel("RMSE")
axes[2].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()